In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import mne
import numpy as np
from numpy.typing import NDArray

from eeg_music.data import EEGMusicDataset
from eeg_music.ica_analysis import (
    apply_ica,
    clean_ica_artifacts,
    ica_band_power_trial,
    ica_band_power_trial_1020,
    windowed_band_power,
    _prepare_raw_1020,
)

In [3]:
bcmi_ds = EEGMusicDataset.load_ondisk(Path("./datasets/bcmi_preprocessed/bcmi_training_export"))
musing_ds = EEGMusicDataset.load_ondisk(Path("./datasets/musin_g_export2"))
print(f"Musing dataset size: {len(musing_ds)} trials")
print(f"Bcmi dataset size: {len(bcmi_ds)} trials")

Musing dataset size: 240 trials
Bcmi dataset size: 3456 trials


In [4]:
from eeg_music.ica_analysis import Normalization, ica_band_power_trial_1020

def make_f(apply_normalization: Normalization = "std"):
  return lambda t : ica_band_power_trial(
    trial=t,
    n_components=8,
    bands = [(0.5, 4), (4, 8), (8, 13), (13, 30), (30, 45)],
    window_sec= 2.0,
    hop_sec = 0.1,
    l_freq = 1.0,
    h_freq = 50.0,
    keep_labels = {"brain", "other"},
    apply_normalization = apply_normalization,
)

In [10]:
from eeg_music.data import MappedDataset

def create_ds(norm):
  return MappedDataset(musing_ds, make_f(norm) )

ds_minmax = create_ds("minmax")
ds_std = create_ds("std")
ds_none = create_ds(None)

In [1]:
from fractions import Fraction
from eeg_music.data import ArrayStratifiedSamplingDataset, temporal_train_test_split
from eeg_music.data import RepeatedDataset
import numpy as np

def create_X_y(dataset):
    X = []
    y = []
    for i in range(len(dataset)):
        sample = dataset[i]
        X.append(sample.eeg_data.get_array().data)
        y.append(sample.music_data.music_id.song_id - 1)
    return np.array(X), np.array(y)

def create_wrapped_ds(ds, repeat=1, trial_length_secs=Fraction(3, 1)):
  return RepeatedDataset(ArrayStratifiedSamplingDataset(ds, 10, trial_length_secs=trial_length_secs), repeat)

def create_split(dataset):
  train_ds, test_ds = temporal_train_test_split(dataset, length_sec=Fraction(20, 1))
  train_ds_repeated = create_wrapped_ds(train_ds, repeat=10)
  test_ds_repeated = create_wrapped_ds(test_ds, repeat=10)
  X_train, y_train = create_X_y(train_ds_repeated)
  X_test, y_test = create_X_y(test_ds_repeated)

  return (X_train, y_train), (X_test, y_test)


/home/zmrocze/studia/uwr/eeg-magisterka/.devenv/state/venv/lib/python3.12/site-packages/speechbrain/utils/torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()


In [ ]:
from fractions import Fraction
from eeg_music.data import ArrayStratifiedSamplingDataset, MappedDataset, std_normalize_trial_channels

def create_split1(dataset):
  def mapped(ds):
    return ArrayStratifiedSamplingDataset(
      MappedDataset(ds, lambda t: make_f("std")(std_normalize_trial_channels((t)))),
      10,
      trial_length_secs=Fraction(3, 1)
      )
  
  splitted = dataset.subject_wise_split(p_train=0.6, p_val=0.0)
  train_ds, test_ds = splitted["train"], splitted["test"]
  train_ds_repeated = mapped(train_ds)
  test_ds_repeated = mapped(test_ds)
  X_train, y_train = create_X_y(train_ds_repeated)
  X_test, y_test = create_X_y(test_ds_repeated)
  return X_train, y_train, X_test, y_test

def create_split2(dataset):
  def mapped(ds):
    return ArrayStratifiedSamplingDataset(
      MappedDataset(ds, lambda t: make_f("std")(std_normalize_trial_channels(t))),
      10,
      trial_length_secs=Fraction(3, 1)
      )
  
  splitted = dataset.subject_wise_split(p_train=0.6, p_val=0.0)
  train_ds, test_ds = splitted["train"], splitted["test"]
  train_ds_repeated = mapped(train_ds)
  test_ds_repeated = mapped(test_ds)
  X_train, y_train = create_X_y(train_ds_repeated)
  X_test, y_test = create_X_y(test_ds_repeated)
  return X_train, y_train, X_test, y_test


In [8]:
split_1 = create_split1(musing_ds)
split_2 = create_split2(musing_ds)

AttributeError: 'OnDiskEeg' object has no attribute 'get_array'

In [ ]:

import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

def experiment(split):
  (X_train, y_train), (X_test, y_test) = split
  # Flatten EEG data for traditional ML models (samples, channels, timepoints) -> (samples, features)
  X_train_flat = X_train.reshape(X_train.shape[0], -1)
  X_test_flat = X_test.reshape(X_test.shape[0], -1)

  print(f"Training set: {X_train_flat.shape}, Labels: {y_train.shape}")
  print(f"Test set: {X_test_flat.shape}, Labels: {y_test.shape}")
  print(f"Number of classes: {len(np.unique(y_train))}")

  for model, model_name in [
    (XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1), "XGBoost"),
    (SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42), "SVM"),
    (KNeighborsClassifier(n_neighbors=5, n_jobs=-1), "KNN")
    ]:

    # Train XGBoost
    print(f"Training {model_name}...")
    model.fit(X_train_flat, y_train)

    # Predict and evaluate
    y_pred_xgb = model.predict(X_test_flat)
    xgb_accuracy = accuracy_score(y_test, y_pred_xgb)

    print(f"{model_name} Test Accuracy: {xgb_accuracy:.4f}")
    print(f"{model_name} Classification Report:")
    print(classification_report(y_test, y_pred_xgb))


In [23]:
for ds, name in [(split_minmax, "Min-Max Normalization"), (split_std, "Standardization"), (split_none, "No Normalization")]:
  print(f"Experiment with {name} dataset:")
  experiment(ds)

Experiment with Min-Max Normalization dataset:
Training set: (24000, 1200), Labels: (24000,)
Test set: (24000, 1200), Labels: (24000,)
Number of classes: 12
Training XGBoost...
XGBoost Test Accuracy: 0.9771
XGBoost Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98      2000
           1       0.95      0.95      0.95      2000
           2       1.00      0.99      1.00      2000
           3       0.96      1.00      0.98      2000
           4       0.91      0.98      0.95      2000
           5       0.99      0.98      0.99      2000
           6       1.00      0.99      1.00      2000
           7       0.98      0.99      0.99      2000
           8       0.98      0.97      0.98      2000
           9       0.98      0.98      0.98      2000
          10       1.00      0.99      0.99      2000
          11       0.99      0.92      0.96      2000

    accuracy                           0.98     24000
   m